# Farm Crop-Mix Optimization Model

**Objective:** Maximize annual farm profit through optimal crop mix across planting successions,
subject to land, water, labor, storage, demand, diversity, and budget constraints.

---

## Model Summary

| Component | Description |
|-----------|-------------|
| **Crops** | Spinach, Cucumbers, Sweet Corn, Napa Cabbage, Hot Peppers, Garlic |
| **Successions** | 14 total (each has a fixed planting calendar across 3–7 months) |
| **Decision variables** | 14 continuous (land fraction per succession) + 14 binary (plant yes/no) |
| **Objective** | Maximize Σ (revenue − variable cost) × x_i − fixed cost |
| **Solver** | `scipy.optimize.milp` (mixed-integer linear program) |

### Constraints

| # | Constraint | Type | Count |
|---|-----------|------|-------|
| 1 | Monthly land usage ≤ 1 (100% of available land) | ≤ | 12 |
| 2 | Monthly labor hours ≤ seasonal labor availability | ≤ | 12 |
| 3 | Monthly water usage ≤ min(catchment, storage×10) | ≤ | 12 |
| 4 | Monthly harvest storage ≤ capacity | ≤ | 12 |
| 5 | Annual yield per crop ≤ annual demand cap | ≤ | 6 |
| 6 | Succession acreage ≤ binary × diversity limit | ≤ | 14 |
| 7 | Total annual cost ≤ budget | ≤ | 1 |

## 1 — Install Dependencies

In [ ]:
import sys
if "google.colab" in sys.modules:
    %pip install scipy numpy pandas matplotlib -q

## 2 — Imports

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import milp, LinearConstraint, Bounds
import matplotlib.pyplot as plt

## 3 — User Inputs

**All parameters below can be edited to match your farm's conditions.**
The solver function in Section 4 reads every value from this section — nothing is hard-coded inside it.

---
### 3A — Farm Infrastructure

In [ ]:
CROPS   = ["Spinach", "Cucumbers", "Sweet Corn", "Napa Cabbage", "Hot Peppers", "Garlic"]
MONTHS  = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
N_CROPS = len(CROPS)

TOTAL_LAND          = 150        # acres
BUDGET              = 3_000_000  # US$/year
HARVEST_STORAGE_CAP = 24_000     # m³  — cool/dry storage capacity
WATER_STORAGE       = 15_000     # liters — on-farm tank capacity
EQUIP_STORAGE       = 30_000     # m³  — barn / equipment shed capacity
FUEL_STORAGE        = 750        # liters — fuel tank capacity

### 3B — Fixed Cost Unit Rates

In [ ]:
# These rates × the capacity parameters above give the annual fixed cost.
# Each rate is multiplied by OPERATING_MONTHS (paid only during the active season).
OPERATING_MONTHS = 11  # months/year the farm incurs fixed costs

RATE_HARVEST_STORAGE  = 2.0    # $/m³/month
RATE_WATER_STORAGE    = 0.002  # $/liter/month
RATE_IRRIGATION       = 175.0  # $/acre/month
RATE_EQUIP_STORAGE    = 2.0    # $/m³/month
RATE_FUEL_STORAGE     = 0.2    # $/liter/month
RATE_MACHINERY        = 2_600  # $/month (fixed machinery lease)

TOTAL_FIXED_COST = (
    HARVEST_STORAGE_CAP * RATE_HARVEST_STORAGE * OPERATING_MONTHS +
    WATER_STORAGE       * RATE_WATER_STORAGE   * OPERATING_MONTHS +
    TOTAL_LAND          * RATE_IRRIGATION      * OPERATING_MONTHS +
    EQUIP_STORAGE       * RATE_EQUIP_STORAGE   * OPERATING_MONTHS +
    FUEL_STORAGE        * RATE_FUEL_STORAGE    * OPERATING_MONTHS +
    RATE_MACHINERY                             * OPERATING_MONTHS
)
print(f"Total annual fixed cost: ${TOTAL_FIXED_COST:,.2f}")

### 3C — Variable Cost Unit Rates

In [ ]:
COST_WATER_PER_L   = 0.002  # $/liter
COST_FERT_PER_LB   = 2.0    # $/lb  (fertilizer + pesticide blended rate)
COST_LABOR_PER_HR  = 20.0   # $/hour (blended wage rate incl. overhead)
COST_FUEL_PER_L    = 1.0    # $/liter

### 3D — Monthly Availability (Water & Labor)

In [ ]:
# Water catchment available each month (liters)
# Effective cap = min(catchment, WATER_STORAGE × 10)
AVAILABLE_WATER_CATCHMENT = np.array(
    [90000, 90000, 157500, 180000, 225000, 225000,
     225000, 225000, 202500, 157500, 112500, 90000], dtype=float
)

# Total seasonal labor hours available each month
AVAILABLE_LABOR = np.array(
    [1280, 1280, 1280, 3840, 3840, 3840,
     7680, 7680, 7680, 7680, 3840, 1280], dtype=float
)

### 3E — Crop Parameters

Values are indexed by crop in this order: `Spinach, Cucumbers, Sweet Corn, Napa Cabbage, Hot Peppers, Garlic`

In [ ]:
#                             Spinach  Cucumbers  Sweet Corn  Napa Cabbage  Hot Peppers  Garlic
YIELD_PER_ACRE        = [    13000,     16000,      10000,       45000,        5000,      11000]  # lbs/acre/harvest-month
CROP_SIZE_M3_PER_LB   = [    0.075,     0.015,      0.015,       0.020,        0.035,     0.025]  # storage m³ per lb
SEED_REQ_PER_ACRE     = [       10,       2.5,         12,        0.75,         0.35,      1000]  # lbs seed per acre
SEED_COST_PER_LB      = [       20,        45,          5,          60,          120,         5]  # $/lb seed
CROP_DIVERSITY        = [     0.80,      0.30,       0.30,        0.50,         0.30,      0.50]  # max land fraction per succession
MONTHLY_DEMAND        = [80156.25,   81562.5,      50000,   130781.25,      57656.25,  102937.5]  # lbs/month demand cap

# Weighted average selling price ($/lb) — accounts for channel mix (wholesale/D2C/B2B)
WEIGHTED_AVG_PRICE = [
    3.4649122807017543,   # Spinach
    1.6293103448275863,   # Cucumbers
    2.275,                # Sweet Corn
    0.728494623655914,    # Napa Cabbage
    2.6585365853658534,   # Hot Peppers
    2.8688524590163933,   # Garlic
]

# Weighted average packaging + SG&A cost ($/lb)
WEIGHTED_AVG_PKG = [
    0.06412280701754386,  # Spinach
    0.04224137931034483,  # Cucumbers
    0.046,                # Sweet Corn
    0.05311827956989247,  # Napa Cabbage
    0.05890243902439024,  # Hot Peppers
    0.038688524590163934, # Garlic
]

# Channel premium multiplier: revenue = yield × price × CHANNEL_PREMIUM
# 1.04 = 80% sold at base price + 20% sold at 120% premium → avg 1.04×
CHANNEL_PREMIUM = 1.04

### 3F — Resource Usage Per Phase Per Crop

Each list has 6 values (one per crop). Rows are growing phases: Planting / Growing / Harvesting.

In [ ]:
# Labor hours per acre per phase-month
#                     Spinach  Cucumbers  Sweet Corn  Napa Cabbage  Hot Peppers  Garlic
LABOR_PLANT = [           15,        15,         10,           10,          20,     20]
LABOR_GROW  = [           12,        12,          6,            6,          15,      8]
LABOR_HARV  = [          100,       130,        200,           80,          80,     75]

# Water liters per acre per phase-month
WATER_PLANT = [         1250,      1750,       1000,         2000,        2000,   1500]
WATER_GROW  = [         1000,       250,        750,         1000,        1250,    500]
WATER_HARV  = [         1125,       750,        875,         1250,         625,   1000]

# Fertilizer lbs per acre per phase-month
FERT_PLANT  = [           15,        12,         20,           15,          10,      8]
FERT_GROW   = [           10,         8,         15,           10,          12,      5]
FERT_HARV   = [            5,         5,          8,            5,           6,      3]

# Fuel liters per acre per phase-month
FUEL_PLANT  = [            8,         8,         10,            8,           7,     10]
FUEL_GROW   = [            3,         2,          4,            3,           3,      2]
FUEL_HARV   = [            6,         5,          8,            5,           4,      5]

### 3G — Succession Calendar

Each succession defines: which crop, which months it occupies (0-indexed Jan=0), and the
growing phase for each of those months (0=Planting, 1=Growing, 2=Harvesting).

To add a new succession, append a dict with the same structure.

In [ ]:
SUCCESSIONS = [
    # ── Spinach (3-month: 1 plant + 1 grow + 1 harvest) ──
    {"name": "Spinach #1",       "crop": 0, "months": [1,2,3],           "phases": [0,1,2]},
    {"name": "Spinach #2",       "crop": 0, "months": [2,3,4],           "phases": [0,1,2]},
    {"name": "Spinach #3",       "crop": 0, "months": [3,4,5],           "phases": [0,1,2]},
    {"name": "Spinach #4",       "crop": 0, "months": [7,8,9],           "phases": [0,1,2]},
    {"name": "Spinach #5",       "crop": 0, "months": [8,9,10],          "phases": [0,1,2]},
    # ── Cucumbers (3-month: 1 plant + 1 grow + 1 harvest) ──
    {"name": "Cucumbers #1",     "crop": 1, "months": [4,5,6],           "phases": [0,1,2]},
    {"name": "Cucumbers #2",     "crop": 1, "months": [5,6,7],           "phases": [0,1,2]},
    {"name": "Cucumbers #3",     "crop": 1, "months": [6,7,8],           "phases": [0,1,2]},
    # ── Sweet Corn (4-month: 1 plant + 2 grow + 1 harvest) ──
    {"name": "Sweet Corn #1",    "crop": 2, "months": [4,5,6,7],         "phases": [0,1,1,2]},
    {"name": "Sweet Corn #2",    "crop": 2, "months": [5,6,7,8],         "phases": [0,1,1,2]},
    # ── Napa Cabbage (4-month: 1 plant + 2 grow + 1 harvest) ──
    {"name": "Napa Cabbage #1",  "crop": 3, "months": [2,3,4,5],         "phases": [0,1,1,2]},
    {"name": "Napa Cabbage #2",  "crop": 3, "months": [7,8,9,10],        "phases": [0,1,1,2]},
    # ── Hot Peppers (7-month: 1 plant + 3 grow + 3 harvest) ──
    {"name": "Hot Peppers #1",   "crop": 4, "months": [3,4,5,6,7,8,9],   "phases": [0,1,1,1,2,2,2]},
    # ── Garlic (7-month: 1 plant + 5 grow + 1 harvest) ──
    {"name": "Garlic #1",        "crop": 5, "months": [0,1,2,3,4,5,6],   "phases": [0,1,1,1,1,1,2]},
]

# Display the planting calendar
_cal = pd.DataFrame("", index=[s["name"] for s in SUCCESSIONS], columns=MONTHS)
_phase_labels = {0: "P", 1: "G", 2: "H"}
for _s in SUCCESSIONS:
    for _m, _p in zip(_s["months"], _s["phases"]):
        _cal.iloc[SUCCESSIONS.index(_s), _m] = _phase_labels[_p]
_cal["Crop"] = [CROPS[s["crop"]] for s in SUCCESSIONS]
_cal

## 4 — Solver Function

In [ ]:
def solve_farm_optimization(
    # ── 3A: Farm infrastructure ──
    total_land          = TOTAL_LAND,
    budget              = BUDGET,
    harvest_storage_cap = HARVEST_STORAGE_CAP,
    water_storage       = WATER_STORAGE,
    equip_storage       = EQUIP_STORAGE,
    fuel_storage        = FUEL_STORAGE,
    # ── 3B: Fixed cost unit rates ──
    operating_months       = OPERATING_MONTHS,
    rate_harvest_storage   = RATE_HARVEST_STORAGE,
    rate_water_storage     = RATE_WATER_STORAGE,
    rate_irrigation        = RATE_IRRIGATION,
    rate_equip_storage     = RATE_EQUIP_STORAGE,
    rate_fuel_storage      = RATE_FUEL_STORAGE,
    rate_machinery         = RATE_MACHINERY,
    # ── 3C: Variable cost unit rates ──
    cost_water_per_l  = COST_WATER_PER_L,
    cost_fert_per_lb  = COST_FERT_PER_LB,
    cost_labor_per_hr = COST_LABOR_PER_HR,
    cost_fuel_per_l   = COST_FUEL_PER_L,
    # ── 3D: Monthly availability ──
    available_water_catchment = AVAILABLE_WATER_CATCHMENT,
    available_labor           = AVAILABLE_LABOR,
    # ── 3E: Crop parameters ──
    yield_per_acre      = YIELD_PER_ACRE,
    crop_size_m3_per_lb = CROP_SIZE_M3_PER_LB,
    seed_req_per_acre   = SEED_REQ_PER_ACRE,
    seed_cost_per_lb    = SEED_COST_PER_LB,
    crop_diversity      = CROP_DIVERSITY,
    monthly_demand      = MONTHLY_DEMAND,
    avg_price           = WEIGHTED_AVG_PRICE,
    avg_pkg             = WEIGHTED_AVG_PKG,
    channel_premium     = CHANNEL_PREMIUM,
    # ── 3F: Resource usage per phase ──
    labor_plant = LABOR_PLANT, labor_grow = LABOR_GROW, labor_harv = LABOR_HARV,
    water_plant = WATER_PLANT, water_grow = WATER_GROW, water_harv = WATER_HARV,
    fert_plant  = FERT_PLANT,  fert_grow  = FERT_GROW,  fert_harv  = FERT_HARV,
    fuel_plant  = FUEL_PLANT,  fuel_grow  = FUEL_GROW,  fuel_harv  = FUEL_HARV,
    # ── 3G: Succession calendar ──
    successions = SUCCESSIONS,
):
    """
    Fully parameterized MILP solver for the farm crop-mix problem.
    Every parameter defaults to the user-input globals defined in Section 3.
    Override any argument to run scenarios without editing Section 3.

    Returns a dict with optimal values, profit breakdown, and solver status.
    """

    n_crops = len(CROPS)
    n_succ  = len(successions)

    # ── Derived inputs ──
    annual_demand_cap  = [d * 11 for d in monthly_demand]
    water_cap_monthly  = np.minimum(available_water_catchment, water_storage * 10)

    labor_by_phase = [labor_plant, labor_grow, labor_harv]
    water_by_phase = [water_plant, water_grow, water_harv]
    fert_by_phase  = [fert_plant,  fert_grow,  fert_harv]
    fuel_by_phase  = [fuel_plant,  fuel_grow,  fuel_harv]

    num_succ_per_crop = [0] * n_crops
    for s in successions:
        num_succ_per_crop[s["crop"]] += 1

    # ── Fixed cost ──
    fixed_cost = (
        harvest_storage_cap * rate_harvest_storage * operating_months +
        water_storage       * rate_water_storage   * operating_months +
        total_land          * rate_irrigation      * operating_months +
        equip_storage       * rate_equip_storage   * operating_months +
        fuel_storage        * rate_fuel_storage    * operating_months +
        rate_machinery                             * operating_months
    )

    # ══════════════════════════════════════════════
    # Variables: x[0..n_succ-1] continuous, y[0..n_succ-1] binary
    # Layout: [x0,...,x_{n-1}, y0,...,y_{n-1}]
    # ══════════════════════════════════════════════
    n_vars = 2 * n_succ

    # ── Objective coefficients (negated for minimization) ──
    c = np.zeros(n_vars)
    var_cost_coeff = np.zeros(n_succ)

    for i, s in enumerate(successions):
        cr     = s["crop"]
        n_harv = sum(1 for p in s["phases"] if p == 2)

        # Revenue per unit x_i (assuming x_i = 1 → full land used)
        rev = n_harv * total_land * yield_per_acre[cr] * avg_price[cr] * channel_premium

        # Variable cost per unit x_i
        seed_c  = total_land * seed_req_per_acre[cr] * seed_cost_per_lb[cr]
        water_c = sum(total_land * water_by_phase[p][cr] for p in s["phases"]) * cost_water_per_l
        fert_c  = sum(total_land * fert_by_phase[p][cr]  for p in s["phases"]) * cost_fert_per_lb
        pkg_c   = n_harv * total_land * yield_per_acre[cr] * avg_pkg[cr]
        fuel_c  = sum(total_land * fuel_by_phase[p][cr]  for p in s["phases"]) * cost_fuel_per_l
        labor_c = sum(total_land * labor_by_phase[p][cr] for p in s["phases"]) * cost_labor_per_hr

        vc = seed_c + water_c + fert_c + pkg_c + fuel_c + labor_c
        var_cost_coeff[i] = vc
        c[i] = -(rev - vc)  # negate: milp minimizes

    # ── Bounds ──
    lb = np.zeros(n_vars)
    ub = np.ones(n_vars)
    integrality = np.zeros(n_vars)
    integrality[n_succ:] = 1  # y variables are binary

    # ══════════════════════════════════════════════
    # Constraints (all ≤ form)
    # ══════════════════════════════════════════════
    A_rows, b_vals = [], []

    # 1. Land: Σ x_i (active in month m) ≤ 1
    for m in range(12):
        row = np.zeros(n_vars)
        for i, s in enumerate(successions):
            if m in s["months"]:
                row[i] = 1.0
        A_rows.append(row); b_vals.append(1.0)

    # 2. Labor: Σ (land × labor_rate) × x_i ≤ available_labor[m]
    for m in range(12):
        row = np.zeros(n_vars)
        for i, s in enumerate(successions):
            if m in s["months"]:
                idx   = s["months"].index(m)
                phase = s["phases"][idx]
                cr    = s["crop"]
                row[i] = total_land * labor_by_phase[phase][cr]
        A_rows.append(row); b_vals.append(available_labor[m])

    # 3. Water: Σ (land × water_rate) × x_i ≤ water_cap_monthly[m]
    for m in range(12):
        row = np.zeros(n_vars)
        for i, s in enumerate(successions):
            if m in s["months"]:
                idx   = s["months"].index(m)
                phase = s["phases"][idx]
                cr    = s["crop"]
                row[i] = total_land * water_by_phase[phase][cr]
        A_rows.append(row); b_vals.append(water_cap_monthly[m])

    # 4. Harvest storage: Σ (land × yield × crop_size) × x_i ≤ harvest_storage_cap
    for m in range(12):
        row = np.zeros(n_vars)
        for i, s in enumerate(successions):
            if m in s["months"]:
                idx = s["months"].index(m)
                if s["phases"][idx] == 2:
                    cr    = s["crop"]
                    row[i] = total_land * yield_per_acre[cr] * crop_size_m3_per_lb[cr]
        A_rows.append(row); b_vals.append(harvest_storage_cap)

    # 5. Annual demand per crop
    for cr_idx in range(n_crops):
        row = np.zeros(n_vars)
        for i, s in enumerate(successions):
            if s["crop"] == cr_idx:
                n_harv = sum(1 for p in s["phases"] if p == 2)
                row[i] = n_harv * total_land * yield_per_acre[cr_idx]
        A_rows.append(row); b_vals.append(annual_demand_cap[cr_idx])

    # 6. Diversity: x_i ≤ y_i × (diversity_limit / num_successions_for_crop)
    for i, s in enumerate(successions):
        cr        = s["crop"]
        div_limit = crop_diversity[cr] / num_succ_per_crop[cr]
        row = np.zeros(n_vars)
        row[i]          =  1.0
        row[n_succ + i] = -div_limit
        A_rows.append(row); b_vals.append(0.0)

    # 7. Budget: Σ var_cost × x_i ≤ budget − fixed_cost
    row = np.zeros(n_vars)
    row[:n_succ] = var_cost_coeff
    A_rows.append(row); b_vals.append(budget - fixed_cost)

    # ── Solve ──
    A = np.array(A_rows)
    b = np.array(b_vals)
    result = milp(
        c           = c,
        constraints = LinearConstraint(A, ub=b),
        integrality = integrality,
        bounds      = Bounds(lb=lb, ub=ub),
        options     = {"disp": False, "presolve": True},
    )

    if not result.success:
        return {"status": "infeasible", "message": result.message}

    x_opt  = result.x[:n_succ]
    y_opt  = result.x[n_succ:]
    profit = -result.fun - fixed_cost

    # ── Build output table ──
    rows = []
    for i, s in enumerate(successions):
        cr     = s["crop"]
        n_harv = sum(1 for p in s["phases"] if p == 2)
        acres      = x_opt[i] * total_land
        yield_lbs  = n_harv * x_opt[i] * total_land * yield_per_acre[cr]
        rev        = yield_lbs * avg_price[cr] * channel_premium
        vc         = var_cost_coeff[i] * x_opt[i]
        rows.append({
            "Succession":        s["name"],
            "Crop":              CROPS[cr],
            "Planted (y)":       int(round(y_opt[i])),
            "Land Fraction (x)": round(x_opt[i], 6),
            "Acres":             round(acres, 2),
            "Yield (lbs)":       round(yield_lbs, 0),
            "Revenue ($)":       round(rev, 2),
            "Variable Cost ($)": round(vc, 2),
            "Contribution ($)":  round(rev - vc, 2),
        })
    detail_df = pd.DataFrame(rows)
    total_revenue  = detail_df["Revenue ($)"].sum()
    total_var_cost = detail_df["Variable Cost ($)"].sum()

    return {
        "status":              "optimal",
        "x":                   x_opt,
        "y":                   y_opt,
        "total_profit":        profit,
        "total_revenue":       total_revenue,
        "total_variable_cost": total_var_cost,
        "total_fixed_cost":    fixed_cost,
        "total_cost":          fixed_cost + total_var_cost,
        "detail":              detail_df,
        "solver_message":      result.message,
    }

## 5 — Solve and Verify Against Excel

In [ ]:
sol = solve_farm_optimization()   # uses all Section 3 defaults
print(f"Status: {sol['status']}")
print(f"Solver: {sol['solver_message']}")
print(f"\nTotal Revenue:       ${sol['total_revenue']:>14,.2f}")
print(f"Total Variable Cost: ${sol['total_variable_cost']:>14,.2f}")
print(f"Total Fixed Cost:    ${sol['total_fixed_cost']:>14,.2f}")
print(f"Total Cost:          ${sol['total_cost']:>14,.2f}")
print(f"Total Profit:        ${sol['total_profit']:>14,.2f}")

In [ ]:
# ── Excel Solver reference values (from Group 5_Baby Model_Final.xlsx) ──
EXCEL_PROFIT     = 6_207_713.07
EXCEL_REVENUE    = 9_080_159.16
EXCEL_TOTAL_COST = 2_872_446.09
EXCEL_FIXED_COST = 1_507_330.00

EXCEL_X = [
    0.11082612, 0.16,       0.0,        0.12637189, 0.05496545,  # Spinach 1-5
    0.10,       0.10,       0.09489313,                          # Cucumbers 1-3
    0.0,        0.09489313,                                      # Sweet Corn 1-2
    0.09489313, 0.11823187,                                      # Napa Cabbage 1-2
    0.21021374,                                                  # Hot Peppers
    0.40,                                                        # Garlic
]

assert sol["status"] == "optimal", "Solver did not find optimal solution"

assert abs(sol["total_profit"]     - EXCEL_PROFIT)     / EXCEL_PROFIT     < 0.01, \
    f"Profit mismatch:     Python ${sol['total_profit']:,.2f}  vs  Excel ${EXCEL_PROFIT:,.2f}"
assert abs(sol["total_fixed_cost"] - EXCEL_FIXED_COST) < 1.0, \
    f"Fixed cost mismatch: Python ${sol['total_fixed_cost']:,.2f}  vs  Excel ${EXCEL_FIXED_COST:,.2f}"
assert abs(sol["total_revenue"]    - EXCEL_REVENUE)    / EXCEL_REVENUE    < 0.01, \
    f"Revenue mismatch:    Python ${sol['total_revenue']:,.2f}  vs  Excel ${EXCEL_REVENUE:,.2f}"
assert abs(sol["total_cost"]       - EXCEL_TOTAL_COST) / EXCEL_TOTAL_COST < 0.01, \
    f"Total cost mismatch: Python ${sol['total_cost']:,.2f}  vs  Excel ${EXCEL_TOTAL_COST:,.2f}"
assert sol["total_cost"] <= BUDGET + 1.0, "Budget constraint violated"

# Verify each succession's land fraction matches Excel within 0.1%
for i, (s, ex_x) in enumerate(zip(SUCCESSIONS, EXCEL_X)):
    py_x = sol["x"][i]
    assert abs(py_x - ex_x) < 0.001, \
        f"Land fraction mismatch for {s['name']}: Python {py_x:.6f} vs Excel {ex_x:.6f}"

print("All assertions passed.")
print(f"\nProfit:  Python ${sol['total_profit']:,.2f}  vs  Excel ${EXCEL_PROFIT:,.2f}  → diff ${abs(sol['total_profit']-EXCEL_PROFIT):.2f}")
print(f"Revenue: Python ${sol['total_revenue']:,.2f}  vs  Excel ${EXCEL_REVENUE:,.2f}  → diff ${abs(sol['total_revenue']-EXCEL_REVENUE):.2f}")
print(f"Cost:    Python ${sol['total_cost']:,.2f}  vs  Excel ${EXCEL_TOTAL_COST:,.2f}  → diff ${abs(sol['total_cost']-EXCEL_TOTAL_COST):.2f}")
print(f"\nAll 14 succession land fractions match Excel within 0.001.")

## 6 — Results

In [ ]:
sol["detail"].style.format({
    "Land Fraction (x)": "{:.4f}",
    "Acres":             "{:.2f}",
    "Yield (lbs)":       "{:,.0f}",
    "Revenue ($)":       "${:,.2f}",
    "Variable Cost ($)": "${:,.2f}",
    "Contribution ($)":  "${:,.2f}",
})

In [ ]:
crop_summary = sol["detail"].groupby("Crop").agg({
    "Acres":             "sum",
    "Yield (lbs)":       "sum",
    "Revenue ($)":       "sum",
    "Variable Cost ($)": "sum",
    "Contribution ($)":  "sum",
}).round(2)
crop_summary.loc["TOTAL"] = crop_summary.sum()
crop_summary.style.format({
    "Acres":             "{:.1f}",
    "Yield (lbs)":       "{:,.0f}",
    "Revenue ($)":       "${:,.2f}",
    "Variable Cost ($)": "${:,.2f}",
    "Contribution ($)":  "${:,.2f}",
})

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
labels = ["Revenue", "Variable Cost", "Fixed Cost", "Profit"]
values = [sol["total_revenue"], -sol["total_variable_cost"],
          -sol["total_fixed_cost"], sol["total_profit"]]
colors = ["#2ecc71", "#e74c3c", "#e67e22", "#3498db"]
ax.bar(labels, values, color=colors, edgecolor="black", linewidth=0.5)
for j, v in enumerate(values):
    ax.text(j, v + 80000 * (1 if v > 0 else -1),
            f"${v/1e6:+.2f}M", ha="center", fontsize=10, fontweight="bold")
ax.set_ylabel("US$")
ax.set_title("Annual Profit Breakdown")
ax.axhline(0, color="black", linewidth=0.8)
plt.tight_layout(); plt.show()

In [ ]:
land_util = pd.DataFrame(0.0, index=[s["name"] for s in SUCCESSIONS], columns=MONTHS)
for i, s in enumerate(SUCCESSIONS):
    for m in s["months"]:
        land_util.iloc[i, m] = sol["x"][i]

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(land_util.values, cmap="YlGn", aspect="auto", vmin=0, vmax=0.5)
ax.set_xticks(range(12)); ax.set_xticklabels(MONTHS)
ax.set_yticks(range(len(SUCCESSIONS))); ax.set_yticklabels([s["name"] for s in SUCCESSIONS])
plt.colorbar(im, ax=ax, label="Land Fraction")
ax.set_title("Monthly Land Allocation by Succession")
totals = land_util.sum(axis=0)
for m in range(12):
    ax.text(m, len(SUCCESSIONS) - 0.3, f"{totals.iloc[m]:.0%}",
            ha="center", va="top", fontsize=8, color="navy", fontweight="bold")
plt.tight_layout(); plt.show()

## 7 — Sensitivity Analysis: Budget Sweep

In [ ]:
budgets  = np.arange(2_000_000, 5_500_000, 250_000)
profits  = []
revenues = []

for b in budgets:
    r = solve_farm_optimization(budget=b)
    profits.append(r["total_profit"]  if r["status"] == "optimal" else np.nan)
    revenues.append(r["total_revenue"] if r["status"] == "optimal" else np.nan)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(budgets/1e6, np.array(profits)/1e6,  "o-",  color="#2ecc71", label="Profit",  linewidth=2)
ax.plot(budgets/1e6, np.array(revenues)/1e6, "s--", color="#3498db", label="Revenue", linewidth=2)
ax.axvline(BUDGET/1e6, color="red", linestyle=":", label=f"Base budget (${BUDGET/1e6:.0f}M)")
ax.set_xlabel("Annual Budget ($M)"); ax.set_ylabel("$M")
ax.set_title("Sensitivity: Profit & Revenue vs. Budget")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

pd.DataFrame({
    "Budget ($M)":  budgets/1e6,
    "Profit ($M)":  np.round(np.array(profits)/1e6, 3),
    "Revenue ($M)": np.round(np.array(revenues)/1e6, 3),
})

In [ ]:
lands        = np.arange(50, 301, 25)
land_profits = []

for la in lands:
    r = solve_farm_optimization(total_land=la)
    land_profits.append(r["total_profit"] if r["status"] == "optimal" else np.nan)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(lands, np.array(land_profits)/1e6, "o-", color="#e67e22", linewidth=2)
ax.axvline(TOTAL_LAND, color="red", linestyle=":", label=f"Base ({TOTAL_LAND} acres)")
ax.set_xlabel("Available Land (acres)"); ax.set_ylabel("Profit ($M)")
ax.set_title("Sensitivity: Profit vs. Available Land")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()